# T16 — Ranking dos quatro modelos por base

A tabela final usa **somente a interseção das datas previstas pelos quatro modelos da mesma base**. As séries completas de `data`, `real` e `previsto` ficam nas saídas dos notebooks de origem. Este notebook lê essas saídas, confere as datas oficiais T08 e o alvo no Excel tratado, une as séries por data e recalcula cada MAE. Nunca compara MAEs de bases distintas, pois as unidades são diferentes.

O ranking descreve os modelos atualmente salvos. As revisões dos MLPs 1 e 5 ocorreram após inspecionar o teste; seus MAEs devem ser relatados como exploratórios.

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preparacao_bases.py').is_file())
BASES = {1: 'bitcoin', 2: 'trafego', 3: 'poluicao', 4: 'clima', 5: 'ouro'}
ARQUIVOS = {'Random Forest': 'random-forest', 'Holt-Winters': 'holt-winters',
            'SARIMAX': 'sarimax', 'MLP Regressor': 'mlp-regressor'}

def saidas_e_execucao(caminho):
    nb = json.loads(caminho.read_text(encoding='utf-8'))
    codigos = [c for c in nb['cells'] if c['cell_type'] == 'code']
    textos = []
    for celula in codigos:
        for saida in celula.get('outputs', []):
            textos.append(''.join(saida.get('text', [])))
            textos.append(''.join(saida.get('data', {}).get('text/plain', [])))
    return '\n'.join(textos), sum(c.get('execution_count') is not None for c in codigos), len(codigos)

def resultado_individual(grupo, modelo):
    caminho = RAIZ / f'grupo{grupo}' / f'{ARQUIVOS[modelo]}-{grupo}.ipynb'
    linha = {'grupo': grupo, 'base': BASES[grupo], 'modelo': modelo,
             'MAE_teste_individual': np.nan, 'MAE_exploratorio': np.nan, 'n_teste': pd.NA,
             'estado': 'notebook ausente', 'origem': ''}
    if not caminho.exists():
        return linha
    linha['origem'] = str(caminho.relative_to(RAIZ))
    texto, executadas, total = saidas_e_execucao(caminho)
    linha['estado'] = f'{executadas}/{total} células executadas'
    marcador = 'HOLT-WINTERS' if modelo == 'Holt-Winters' else 'SARIMAX'
    mae_oficial = re.search(
        rf'MAE OFICIAL T08 {marcador} {BASES[grupo]}:\s*([\d.]+)\s*\|\s*n=([\d,]+)', texto
    ) if modelo in ('Holt-Winters', 'SARIMAX') else None
    if mae_oficial:
        linha['MAE_teste_individual'] = float(mae_oficial.group(1))
        linha['n_teste'] = int(mae_oficial.group(2).replace(',', ''))
        if modelo == 'SARIMAX' and grupo in (2, 3):
            mae_antigo = re.search(r'MAE de teste:\s*([\d.]+)', texto)
            if mae_antigo:
                linha['MAE_exploratorio'] = float(mae_antigo.group(1))
            linha['estado'] += '; grade completa, seção T08 corrigida'
        if modelo == 'Holt-Winters':
            linha['estado'] += '; seção T08 de um passo; estudo multipasso acima é adicional'
        return linha
    if modelo == 'Holt-Winters':
        linha['estado'] += '; seção oficial T08 pendente de execução'
        return linha
    if modelo == 'Random Forest':
        if grupo == 4:
            achado = re.search(r'^\s*0\s+Random Forest\s+([\d,]+)\s+([\d.]+)', texto, re.M)
            if achado: n, mae = achado.groups()
        else:
            mae_encontrado = re.search(r'MAE RF:\s*([\d.]+)', texto)
            n_encontrado = re.search(r'previsões:\s*([\d,]+)', texto)
            achado = bool(mae_encontrado and n_encontrado)
            if achado: mae, n = mae_encontrado.group(1), n_encontrado.group(1)
    elif modelo == 'MLP Regressor':
        achado = re.search(r'teste_final\s+([\d,]+)\s+([\d.]+)', texto)
        if achado: n, mae = achado.groups()
    else:
        mae_encontrado = re.search(r'MAE de teste:\s*([\d.]+)', texto)
        n_encontrado = re.search(r'Previsões de um(?:a)? [^:]+:\s*([\d,]+)', texto)
        achado = bool(mae_encontrado and n_encontrado)
        if achado: mae, n = mae_encontrado.group(1), n_encontrado.group(1)
    if achado:
        linha['MAE_teste_individual'] = float(mae)
        linha['n_teste'] = int(n.replace(',', ''))
        if executadas < total:
            linha['estado'] += '; execução parcial — conferir reprodução'
        if modelo == 'MLP Regressor' and grupo in (1, 5):
            linha['estado'] += '; revisão após ver teste — MAE exploratório'
        if modelo == 'SARIMAX' and grupo in (2, 3):
            linha['MAE_exploratorio'] = linha['MAE_teste_individual']
            linha['MAE_teste_individual'] = np.nan
            cobertura = {2: '7.720/10.267', 3: '3.023/6.336'}[grupo]
            linha['estado'] += f'; lacunas comprimidas e {cobertura} origens'
    return linha

resultados = pd.DataFrame([
    resultado_individual(grupo, modelo)
    for grupo in BASES for modelo in ARQUIVOS
])
display(resultados.round(4))
print(f'MAEs individuais disponíveis: {resultados.MAE_teste_individual.notna().sum()}/20')

,grupo,base,modelo,MAE_teste_individual,MAE_exploratorio,n_teste,estado,origem
0,1,bitcoin,Random Forest,1661.4218,NaN,884,10/10 células executadas,grupo1\random-forest-1.ipynb
1,1,bitcoin,Holt-Winters,1404.1918,NaN,884,15/15 células executadas; seção T08 de um pass...,grupo1\holt-winters-1.ipynb
2,1,bitcoin,SARIMAX,1391.9008,NaN,884,23/23 células executadas,grupo1\sarimax-1.ipynb
3,1,bitcoin,MLP Regressor,1401.8277,NaN,884,18/18 células executadas; revisão após ver tes...,grupo1\mlp-regressor-1.ipynb
4,2,trafego,Random Forest,153.2877,NaN,10267,13/13 células executadas,grupo2\random-forest-2.ipynb
5,2,trafego,Holt-Winters,332.7024,NaN,10267,15/15 células executadas; seção T08 de um pass...,grupo2\holt-winters-2.ipynb
6,2,trafego,SARIMAX,351.8835,295.6266,10267,"22/23 células executadas; grade completa, seçã...",grupo2\sarimax-2.ipynb
7,2,trafego,MLP Regressor,180.4071,NaN,10267,18/18 células executadas,grupo2\mlp-regressor-2.ipynb
8,3,poluicao,Random Forest,9.8670,NaN,6336,14/14 células executadas,grupo3\random-forest-3.ipynb
9,3,poluicao,Holt-Winters,10.3912,NaN,6336,7/15 células executadas; seção T08 de um passo...,grupo3\holt-winters-3.ipynb


MAEs individuais disponíveis: 20/20


## Comparação na interseção das datas

O bloco abaixo só gera ranking quando os quatro notebooks de uma base fornecem previsões completas. Ele verifica datas únicas e ordenadas, pertencimento ao T08, alvos iguais aos do Excel tratado e coerência do MAE recalculado com o valor individual salvo. A cobertura é medida sobre a grade inteira do teste.

In [2]:
TIPO_PREVISOES = 'application/vnd.series-temporais.previsoes+json'
import base64
import zlib
from preparacao_bases import carregar_base_tratada
from walk_forward import origens_teste

def previsoes_do_notebook(grupo, modelo):
    caminho = RAIZ / f'grupo{grupo}' / f'{ARQUIVOS[modelo]}-{grupo}.ipynb'
    if not caminho.exists():
        return None
    nb = json.loads(caminho.read_text(encoding='utf-8'))
    cargas = [saida.get('data', {}).get(TIPO_PREVISOES)
              for celula in nb['cells'] for saida in celula.get('outputs', [])]
    cargas = [c for c in cargas if c is not None]
    if not cargas:
        return None
    if len(cargas) != 1:
        raise ValueError(f'{caminho}: esperada uma única série completa de previsões')
    carga = cargas[0]
    if isinstance(carga, dict) and carga.get('formato') == 'json-zlib-base64':
        carga = json.loads(zlib.decompress(base64.b64decode(carga['dados'])).decode('utf-8'))
    dados = pd.DataFrame(carga)
    if set(dados.columns) != {'data', 'real', 'previsto'}:
        raise ValueError(f'{caminho}: colunas de previsão inválidas')
    dados['data'] = pd.to_datetime(dados['data'])
    dados = dados.set_index('data').sort_index()
    if dados.index.has_duplicates or dados.isna().any().any():
        raise ValueError(f'{caminho}: datas duplicadas ou valores faltantes')
    if not np.isfinite(dados[['real', 'previsto']].to_numpy(dtype=float)).all():
        raise ValueError(f'{caminho}: previsão não finita')
    return dados

comparacoes = []
pendencias = []
resumo_bases = []
for grupo, base in BASES.items():
    series = {modelo: previsoes_do_notebook(grupo, modelo) for modelo in ARQUIVOS}
    faltantes = [modelo for modelo, serie in series.items() if serie is None]
    if faltantes:
        pendencias.append({'grupo': grupo, 'base': base,
                           'pendencia': 'previsões completas: ' + ', '.join(faltantes)})
        continue
    oficiais = origens_teste(base, RAIZ)
    for modelo, serie in series.items():
        fora = serie.index.difference(oficiais)
        if len(fora):
            raise ValueError(f'Grupo {grupo}, {modelo}: {len(fora)} datas fora do T08')
    datas_comuns = oficiais
    for serie in series.values():
        datas_comuns = datas_comuns.intersection(serie.index)
    if datas_comuns.empty:
        raise ValueError(f'Grupo {grupo}: interseção de datas vazia')
    df_base, meta = carregar_base_tratada(base, RAIZ)
    alvo_verdadeiro = df_base.loc[datas_comuns, meta['alvo']].to_numpy(dtype=float)
    if not np.isfinite(alvo_verdadeiro).all():
        raise ValueError(f'Grupo {grupo}: alvo ausente na interseção')
    for modelo, serie in series.items():
        recorte = serie.loc[datas_comuns]
        np.testing.assert_allclose(recorte.real.to_numpy(dtype=float), alvo_verdadeiro,
                                   rtol=0, atol=1e-8,
                                   err_msg=f'Grupo {grupo}, {modelo}: alvo divergente do Excel')
        mae = float(np.abs(alvo_verdadeiro - recorte.previsto.to_numpy(dtype=float)).mean())
        individual = resultados.loc[(resultados.grupo == grupo) &
                                   (resultados.modelo == modelo)].iloc[0]
        if len(serie) == int(individual.n_teste):
            mae_total = float(np.abs(serie.real - serie.previsto).mean())
            if not np.isclose(mae_total, individual.MAE_teste_individual, atol=5e-4, rtol=0):
                raise ValueError(f'Grupo {grupo}, {modelo}: MAE recalculado {mae_total:.6f} '
                                 f'difere do valor salvo {individual.MAE_teste_individual:.6f}')
        comparacoes.append({'grupo': grupo, 'base': base, 'modelo': modelo,
                            'n_previsto': len(serie), 'n_datas_comuns': len(datas_comuns),
                            'inicio': datas_comuns.min(), 'fim': datas_comuns.max(),
                            'MAE_na_intersecao': mae})
    resumo_bases.append({'grupo': grupo, 'base': base,
                         'n_grade_teste': int(meta['linhas_teste']),
                         'n_origens_T08': len(oficiais),
                         'n_datas_comuns': len(datas_comuns),
                         'cobertura_grade_pct': 100 * len(datas_comuns) / meta['linhas_teste']})
if pendencias:
    display(pd.DataFrame(pendencias))
if comparacoes:
    comparacao = pd.DataFrame(comparacoes)
    comparacao['posicao_na_base'] = comparacao.groupby('grupo').MAE_na_intersecao.rank(method='min').astype(int)
    resumo_cobertura = pd.DataFrame(resumo_bases)
    display(resumo_cobertura.round(2))
    display(comparacao.sort_values(['grupo', 'posicao_na_base']).round(4))
    vencedores = comparacao.loc[comparacao.groupby('grupo').MAE_na_intersecao.idxmin()]
    display(vencedores[['base', 'modelo', 'MAE_na_intersecao', 'n_datas_comuns']].round(4))
else:
    print('Ranking pendente: nenhuma base possui as quatro séries completas de previsão.')


,grupo,base,pendencia
0,4,clima,previsões completas: SARIMAX


,grupo,base,n_grade_teste,n_origens_T08,n_datas_comuns,cobertura_grade_pct
0,1,bitcoin,884,884,884,100.00
1,2,trafego,10511,10267,10267,97.68
2,3,poluicao,7013,6336,6336,90.35
3,5,ouro,601,601,601,100.00


,grupo,base,modelo,n_previsto,n_datas_comuns,inicio,fim,MAE_na_intersecao,posicao_na_base
2,1,bitcoin,SARIMAX,884,884,2024-04-13 12:00:00,2026-09-13 12:00:00,1391.9008,1
3,1,bitcoin,MLP Regressor,884,884,2024-04-13 12:00:00,2026-09-13 12:00:00,1401.8277,2
1,1,bitcoin,Holt-Winters,884,884,2024-04-13 12:00:00,2026-09-13 12:00:00,1404.1918,3
0,1,bitcoin,Random Forest,884,884,2024-04-13 12:00:00,2026-09-13 12:00:00,1661.4218,4
4,2,trafego,Random Forest,10267,10267,2017-07-20 01:00:00,2018-09-30 23:00:00,153.2877,1
7,2,trafego,MLP Regressor,10267,10267,2017-07-20 01:00:00,2018-09-30 23:00:00,180.4071,2
5,2,trafego,Holt-Winters,10267,10267,2017-07-20 01:00:00,2018-09-30 23:00:00,332.7024,3
6,2,trafego,SARIMAX,10267,10267,2017-07-20 01:00:00,2018-09-30 23:00:00,351.8835,4
8,3,poluicao,Random Forest,6336,6336,2016-05-12 19:00:00,2017-02-28 23:00:00,9.8670,1
9,3,poluicao,Holt-Winters,6336,6336,2016-05-12 19:00:00,2017-02-28 23:00:00,10.3912,2


,base,modelo,MAE_na_intersecao,n_datas_comuns
2,bitcoin,SARIMAX,1391.9008,884
4,trafego,Random Forest,153.2877,10267
8,poluicao,Random Forest,9.8670,6336
14,ouro,SARIMAX,19.5104,601


### Como ler o ranking

Menor MAE é melhor **dentro da mesma base e nas mesmas datas**. A cobertura mostra quantos instantes da grade original entraram na comparação. Os modelos podem ter políticas diferentes de reajuste; o que é comum é o horizonte de um passo, o alvo e o conjunto de datas. O SARIMAX 4 ainda exibe avisos de convergência; os MLPs 1 e 5 sofreram revisão depois de ver o teste. Essas ressalvas acompanham a interpretação do resultado, mesmo quando o cálculo da interseção está correto.

Consulte `GUIA_PRODUCAO_E_STATUS.md` para o fluxo de produção e os próximos passos do relatório.

## Ranking visual

Cada célula mostra o MAE dividido pelo menor MAE **da própria base**. `1,00×` é o melhor resultado daquela base; valores maiores indicam mais erro. As unidades originais permanecem na tabela acima.

In [ ]:
import matplotlib.pyplot as plt
if len(resumo_bases) == len(BASES):
    ordem_modelos = list(ARQUIVOS)
    ordem_bases = [BASES[g] for g in BASES]
    matriz = comparacao.pivot(index='base', columns='modelo', values='MAE_na_intersecao').loc[ordem_bases, ordem_modelos]
    relativos = matriz.div(matriz.min(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    imagem = ax.imshow(relativos, cmap='YlGnBu', vmin=1, vmax=max(2, float(relativos.max().max())))
    ax.set_xticks(range(len(ordem_modelos)), ordem_modelos)
    ax.set_yticks(range(len(ordem_bases)), ordem_bases)
    for i in range(len(ordem_bases)):
        for j in range(len(ordem_modelos)):
            ax.text(j, i, f'{relativos.iloc[i, j]:.2f}×', ha='center', va='center',
                    color='white' if relativos.iloc[i, j] > 1.65 else 'black')
    ax.set_title('Erro relativo ao melhor modelo de cada base (mesmas datas)')
    fig.colorbar(imagem, ax=ax, label='MAE / menor MAE da base')
    plt.tight_layout()
    plt.show()
else:
    print('Gráfico completo disponível após as cinco bases terem quatro séries de previsões.')
